# GPU tier smoke testRun this on Kaggle with a **GPU accelerator and Internet enabled** beforecommitting GPU hours to the twelve outstanding models.The point is to find out what breaks. Nothing here is a benchmark run: oneticker, ten sessions, one epoch. Every cell is independently runnable — eachcarries its own imports, its own data, and its own installs — so a cell thatfails can be re-run alone after a fix without replaying the notebook.Each cell ends in a single `PASS` or `FAIL` line. Cell 3 is expected to be thenoisy one; it reports and does not debug.**Ordering is deliberate.** Kronos runs first because it is the riskiest: aseparate repository rather than a pip package, a separate API, its owntokeniser, and OHLCV input where everything else takes a univariate series. Ifit fails, that is worth knowing before any budget is spent.Reference: `configs/models.yaml` and `src/sp500bench/models/registry.py` are thesource of truth for checkpoints. The five foundation entries are Chronos-2,TimesFM-2.5, Moirai-2.0, Kronos-small and TTM.

## What the smoke test has establishedKaggle, Tesla T4 15.6 GB, torch 2.10.0+cu128, CUDA 12.8. Five runs so far.| model | status | detail ||---|---|---|| **Kronos-small** | **PASS** | loads and runs || **Moirai-2.0** | **LOADED** | predictor constructs || **TimesFM-2.5** | **RESOLVED** | entry point is `timesfm.TimesFM3Forecaster` || Chronos-2 | **UNTESTED** | never resolved on its own — collateral in a joint transaction || TTM | **UNTESTED** | same || Moirai-2.0 | **LOADED** once, then untested | same collateral || timesfm | **CONFLICT** | `timesfm==1.0.0 ... ResolutionImpossible` — the only demonstrated incompatibility || environment | **SOLVED** | pins held: numpy 2.0.2, scipy 1.16.3, pandas 2.3.3, torch stack || DLinear (cell 4) | fixed, not yet re-run | wrong argument name, corrected |### The constraint works. Five runs in, here is what is actually known.**The generated pins held.** On the last run `torch`, `torchvision`,`torchaudio` all stayed put, and so did **numpy 2.0.2, scipy 1.16.3, pandas2.3.3**. No ABI break. `transformers` pinned to the 4.x line as intended. Theenvironment problem is solved; what remains is a question about one package.Getting there took two layers, and they were one bug:| layer | what moved | how it surfaced ||---|---|---|| 1 | torch 2.10.0+cu128 → 2.14.0+cu130 | `ImportError: cannot import name 'PreTrainedModel' from 'transformers'` — under a transformers version where that symbol exists || 2 | numpy → 1.26.4, scipy → 1.11.4 | `ValueError: numpy.dtype size changed, Expected 96 from C header, got 88` on `import pandas` |Both are a dependency quietly moving a binary-compiled package out from undersomething compiled against it. The fix is to pin **what the image actuallyhas**, read immediately before installing, rather than a hardcoded list —so a change to Kaggle's base image cannot silently reintroduce it. The localversion suffix is stripped (`2.10.0+cu128` → `torch==2.10.0`), which underPEP 440 still matches the installed build while rejecting `2.14.0+cu130`.### Only timesfm is demonstrably in conflictThe four packages were previously installed in **one** pip call. pip resolves asingle call jointly, so one unresolvable package fails the whole transaction and**none** install. The output named exactly one culprit:```Cannot install timesfm==1.0.0 ... ResolutionImpossible```but the cell reported all four as incompatible. `chronos-forecasting`, `uni2ts`and `granite-tsfm` were **collateral** — they were never tested under theconstraint at all. Recording "all four are incompatible" would have put anunverified claim in the write-up.Cell 3 now installs **each package in its own transaction**, against the sameconstraints file, and records each outcome separately: installed or not, andpip's own words when it fails. Nothing is quiet — the resolver's explanation isthe evidence, captured to `/tmp/pip-<package>.log`. Guard 2 names the specificpackages that failed and says explicitly that nothing there is collateral.Models whose package installed are loaded; the rest are marked **SKIPPED** withthe reason, which is a statement about the package and not about the checkpoint.**The pins are not widened to accommodate a package.** A package that cannotinstall against the image's own versions is a genuine finding about thatpackage, and gets recorded as one — not worked around by letting numpy or torchmove and breaking something else downstream.### Restarting the kernel is not enough — stop the session**A kernel restart does not reset pip state on Kaggle.** It clears Python stateand leaves `site-packages` exactly as the previous cell's pip left it. After anunconstrained install, restarting gives a fresh interpreter on top of a brokenenvironment: the symptoms move and the cause does not.**Use the power button and stop the session entirely** to get the clean imageback. Then run cell 1 and the cell you are testing — nothing else. Cell 2imports transformers, and pip cannot replace a module that is already imported.This is also why the second run's transformers pin was never exercised: pip put4.56.0 on disk while the kernel kept serving 5.0.0.### Cell 4 — wrong argument, now corrected```TypeError: Trainer.__init__() got an unexpected keyword argument'num_workers_loader'````num_workers_loader` does not exist in neuralforecast 3.2.1. Anything the modelconstructor does not name falls into `**trainer_kwargs` and is passed to`pl.Trainer`, which rejects it. Verified against the 3.2.1 source rather thanassumed: `grep -rn num_workers_loader neuralforecast/` returns nothing, and`DLinear.__init__` names `dataloader_kwargs=None`. The correct call is`DLinear(dataloader_kwargs={"num_workers": 0}, accelerator="gpu", devices=1)` —dataloader settings in the dict, trainer settings loose.Zero workers remains a workaround for cell ordering, not a fix. **The realneural-tier run belongs in its own session.**### Two things Kronos passing has already settledBoth are already in `registry.py` and `configs/models.yaml`; the smoke testconfirms them against the running model.- It emits **price-level OHLCV**, so returns must be derived from the predicted  close rather than read off directly.- It samples autoregressively, so its **CRPS must come from sampled paths**, not  a closed-form quantile head. `sample_count=5` here is a smoke test; a real run  needs far more, and that is a compute cost the budget has to carry.### Suggested next runStop the session between each of these, not just restart the kernel.1. Cell 1, then **cell 3 alone** — per-package installs. Settles whether   Chronos-2, Moirai-2.0 and TTM resolve on their own, which is the only open   question left. timesfm is expected to fail; that outcome is already known.2. Cell 1, then cell 2 — reconfirms Kronos.3. Cell 1, then cell 4 — confirms the nine quantile columns.

In [ ]:
# ============================================================# CELL 1 - ENVIRONMENT# ============================================================!pip install -q neuralforecast statsforecast pyyamlimport sysprint("=" * 60)ok = Truetry:    import torch    print(f"torch              {torch.__version__}")    cuda = torch.cuda.is_available()    print(f"CUDA available     {cuda}")    if cuda:        print(f"GPU                {torch.cuda.get_device_name(0)}")        print(f"CUDA version       {torch.version.cuda}")        total = torch.cuda.get_device_properties(0).total_memory / 1e9        print(f"GPU memory         {total:.1f} GB")    else:        print("GPU                none - accelerator is off or unavailable")        ok = Falseexcept Exception as exc:    print(f"torch import failed: {type(exc).__name__}: {exc}")    ok = Falsefor name in ("neuralforecast", "statsforecast", "yaml"):    try:        module = __import__(name)        print(f"{name:<19}{getattr(module, '__version__', 'ok')}")    except Exception as exc:        print(f"{name:<19}FAILED: {type(exc).__name__}: {exc}")        ok = Falseprint(f"python             {sys.version.split()[0]}")print("=" * 60)print("PASS - environment ready" if ok else      "FAIL - no GPU or a core package is missing; fix before running cells 2-4")

In [ ]:
# ============================================================# CELL 2 - KRONOS  (riskiest first: separate repo, separate API)# ============================================================# This cell imports transformers. If you intend to run cell 3 afterwards, stop# the session first (power button) - a kernel restart clears Python state but# leaves pip's site-packages as it was, so cell 3's constraints cannot take# effect on top of it.# ============================================================import sysimport warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")TICKER = "AAPL"CONTEXT = 400          # Kronos-small documents a 512-session context limitPRED_LEN = 10SAMPLE_COUNT = 5# Idempotent: the guard is shell-side so the cell can be re-run alone# without git erroring on an existing directory.![ -d Kronos ] || git clone --depth 1 https://github.com/shiyu-coder/Kronos.gitif "Kronos" not in sys.path:    sys.path.append("Kronos")def load_bars(ticker, n):    """OHLCV bars. Kronos is the one model here that takes them natively."""    try:        import yfinance as yf        frame = yf.download(ticker, period="3y", progress=False, auto_adjust=True)        if len(frame):            if isinstance(frame.columns, pd.MultiIndex):                frame.columns = frame.columns.get_level_values(0)            frame = frame.rename(columns=str.lower)[                ["open", "high", "low", "close", "volume"]].dropna()            return frame.tail(n), "yfinance"    except Exception as exc:        print(f"  yfinance unavailable ({type(exc).__name__}); using synthetic bars")    rng = np.random.default_rng(20260902)    close = 100 * np.exp(np.cumsum(rng.normal(0, 0.015, n)))    frame = pd.DataFrame({        "open": close * (1 + rng.normal(0, 0.002, n)),        "high": close * (1 + abs(rng.normal(0, 0.006, n))),        "low": close * (1 - abs(rng.normal(0, 0.006, n))),        "close": close,        "volume": rng.integers(1e6, 5e6, n).astype(float),    }, index=pd.bdate_range("2023-01-02", periods=n))    return frame, "synthetic"print("=" * 60)try:    from model import Kronos, KronosPredictor, KronosTokenizer    tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")    model = Kronos.from_pretrained("NeoQuasar/Kronos-small")    print("checkpoint + tokenizer loaded")    import torch    device = "cuda:0" if torch.cuda.is_available() else "cpu"    predictor = KronosPredictor(model, tokenizer, device=device, max_context=512)    bars, source = load_bars(TICKER, CONTEXT + PRED_LEN)    history, future = bars.iloc[:-PRED_LEN], bars.iloc[-PRED_LEN:]    print(f"data               {source}, {len(history)} context sessions")    out = predictor.predict(        df=history[["open", "high", "low", "close", "volume"]],        x_timestamp=pd.Series(history.index),        y_timestamp=pd.Series(future.index),        pred_len=PRED_LEN,        T=1.0,        top_p=0.9,        sample_count=SAMPLE_COUNT,        verbose=False,    )    print(f"device             {device}")    print(f"output type        {type(out).__name__}")    print(f"output shape       {getattr(out, 'shape', 'n/a')}")    print(f"output columns     {list(getattr(out, 'columns', []))}")    print(out.head(3) if hasattr(out, "head") else out)    print()    print("NOTE: CRPS for Kronos must come from sampled paths, not a quantile")    print(f"head. sample_count={SAMPLE_COUNT} here; a real run needs far more.")    print("=" * 60)    print(f"PASS - Kronos ran, {PRED_LEN} sessions x {SAMPLE_COUNT} samples")except Exception as exc:    import traceback    traceback.print_exc()    print("=" * 60)    print(f"FAIL - Kronos: {type(exc).__name__}: {exc}")

In [ ]:
# ============================================================# CELL 3 - THE OTHER FOUR FOUNDATION CHECKPOINTS# One load + one call each. Report only - do not debug here.# ============================================================# ** STOP THE SESSION BEFORE RUNNING THIS CELL. **## Not "restart the kernel" - that clears Python state but leaves site-packages# as a previous cell's pip left it. Only stopping the session (the power button)# restores the clean image. Then run cell 1 and this cell only: cell 2 imports# transformers, and pip cannot replace an already-imported module.# ============================================================# WHY THE CONSTRAINTS FILE IS GENERATED, in two layers.## Layer 1, torch. Unconstrained, these installs drag torch from 2.10.0+cu128 to# 2.14.0+cu130. Kaggle's torchvision and torchaudio are compiled against# 2.10.0, so they land partially initialised; transformers imports torch and# breaks with it, surfacing as#     ImportError: cannot import name 'PreTrainedModel' from 'transformers'# under a transformers version where that symbol certainly exists.## Layer 2, numpy. Pinning only torch let the same failure through one level# down: numpy went to 1.26.4 and scipy to 1.11.4 against an image shipping# numpy 2.x, and pandas' C extension broke with#     ValueError: numpy.dtype size changed, Expected 96 from C header, got 88## So the pins are generated from whatever the image actually has, immediately# before installing. Nothing is widened to accommodate a package: a package that# cannot install against the image's own versions is a finding about that# package, and is recorded as one.# ============================================================# WHY EACH PACKAGE INSTALLS IN ITS OWN TRANSACTION.## They were previously installed in one pip call. pip resolves a single call# jointly, so one unresolvable package fails the whole transaction and NONE# install. The run that failed named exactly one culprit -#     Cannot install timesfm==1.0.0 ... ResolutionImpossible# - but the cell reported all four as incompatible. Three of them were# collateral and had never been tested under the constraint. Recording that as# "all four are incompatible" would have put an unverified claim in the# write-up.## One package per call, each against the same constraints file, each recorded# on its own. Not quiet: the resolver's explanation is the evidence for the# write-up, so it is captured to /tmp/pip-<package>.log and the failing extract# is printed.# ============================================================import pathlibimport reimport subprocessimport sysfrom importlib.metadata import PackageNotFoundErrorfrom importlib.metadata import version as dist_version# Reported before and after. transformers is WATCHED, not pinned - it is the one# package this cell deliberately moves (to the 4.x line).STACK = ("torch", "torchvision", "torchaudio", "numpy", "scipy", "pandas",         "transformers")# Pinned to their observed versions. Each is a compiled extension, or something# compiled extensions link against, so any of them moving breaks something else.PIN = ("torch", "torchvision", "torchaudio", "numpy", "scipy", "pandas")# model name -> the distribution that provides itPACKAGE_FOR_MODEL = {    "Chronos-2": "chronos-forecasting",    "TimesFM-2.5": "timesfm",    "Moirai-2.0": "uni2ts",    "TTM": "granite-tsfm",}WANTED = list(PACKAGE_FOR_MODEL.values())CONSTRAINTS = "/tmp/c.txt"def versions():    out = {}    for name in STACK:        try:            out[name] = dist_version(name)        except PackageNotFoundError:            out[name] = None    return outbefore = versions()print("BEFORE install")for name in STACK:    print(f"  {name:<16}{before[name] or 'not installed'}")print()# Generate the pins from what is actually here. The local version suffix is# stripped - "2.10.0+cu128" becomes "2.10.0" - because a constraint naming a# local build can only be satisfied by an index publishing that exact build.# Under PEP 440 a public-version specifier still matches the local one, so# torch==2.10.0 is satisfied by the installed 2.10.0+cu128 and nothing moves,# while 2.14.0+cu130 is correctly rejected.pins = []for name in PIN:    observed = before[name]    if observed:        pins.append(f"{name}=={observed.split('+')[0]}")    else:        print(f"  note: {name} not installed, so not pinned")pathlib.Path(CONSTRAINTS).write_text("\n".join(pins) + "\n")print(f"--- {CONSTRAINTS} (generated from the image) ---")print(pathlib.Path(CONSTRAINTS).read_text(), end="")print("------------------------------------------------")print()def pip_install(package):    """One package, one transaction, full resolver output kept.    Returns (ok, reason, log_path). No -q: when a resolve fails, pip's own    explanation is the evidence, and a summarised version of it is not.    """    slug = re.sub(r"[^A-Za-z0-9._-]", "_", package)    log = pathlib.Path(f"/tmp/pip-{slug}.log")    proc = subprocess.run(        [sys.executable, "-m", "pip", "install", "-c", CONSTRAINTS, package],        capture_output=True, text=True)    log.write_text(proc.stdout + "\n" + proc.stderr)    if proc.returncode == 0:        return True, "", log    combined = (proc.stdout + "\n" + proc.stderr).splitlines()    signal = [ln for ln in combined              if any(k in ln for k in ("ResolutionImpossible", "Cannot install",                                       "conflict is caused by", "The user requested",                                       "depends on", "ERROR:"))]    reason = " | ".join(signal[:4]) if signal else combined[-1] if combined else "unknown"    return False, reason, logprint("INSTALLING - one package per transaction, same constraints file")print("=" * 60)install_results = {}for package in WANTED:    ok, reason, log = pip_install(package)    install_results[package] = (ok, reason, log)    status = "OK    " if ok else "FAILED"    print(f"  {status} {package:<22} log: {log}")    if not ok:        print(f"         reason (pip's own words):")        for line in reason.split(" | "):            print(f"           {line[:150]}")print("=" * 60)print()# transformers is moved deliberately, and still under the same constraints so it# cannot drag anything pinned along with it.tf_ok, tf_reason, tf_log = pip_install("transformers<5")print(f"transformers<5     {'OK' if tf_ok else 'FAILED'}   log: {tf_log}")if not tf_ok:    print(f"  {tf_reason[:200]}")print()after = versions()print("AFTER install")for name in STACK:    moved = "" if before[name] == after[name] else f"   <-- MOVED from {before[name]}"    print(f"  {name:<16}{after[name] or 'not installed'}{moved}")print()# --- Guard 1: nothing in PIN may have moved ----------------------------------moved = [n for n in PIN if before[n] is not None and before[n] != after[n]]if moved:    print("=" * 60)    print(f"FAIL - the constraint did not hold; {', '.join(moved)} moved.")    print("       Everything below would be testing a broken environment.")    raise SystemExit(f"constrained packages moved: {', '.join(moved)}")print(f"Guard 1  PASS - all {len(PIN)} pinned packages held")# --- Guard 2: per-package outcome, no collateral blame ------------------------# Each package had its own transaction, so a failure here is a statement about# that package and nothing else.installed, failed = [], []for package in WANTED:    try:        dist_version(package)        installed.append(package)    except PackageNotFoundError:        failed.append(package)if failed:    print(f"Guard 2  {len(failed)} of {len(WANTED)} did not install under the constraint:")    for package in failed:        _, reason, log = install_results[package]        print(f"           {package} - {reason.split(' | ')[0][:120]}")        print(f"             full resolver output: {log}")    print("         This is a finding about those packages, not about the")    print("         constraint. Each was resolved on its own, so nothing here")    print("         is collateral from another package's failure. The pins are")    print("         NOT widened to accommodate them.")else:    print(f"Guard 2  PASS - all {len(WANTED)} installed under the constraint")if not installed:    print("=" * 60)    print("FAIL - nothing installed; there is no model to load.")    raise SystemExit("no model package installed under the constraint")# --- Guard 3: the numpy/pandas ABI must actually work ------------------------# Matching version numbers is necessary but not sufficient: what breaks is the# compiled extension, and the only honest check is to import it and use it.try:    import numpy    import pandas    _probe = pandas.DataFrame({"x": numpy.arange(3, dtype="float64")})    assert float(_probe["x"].sum()) == 3.0    print(f"Guard 3  PASS - numpy {numpy.__version__} / pandas "          f"{pandas.__version__} import and interoperate")except Exception as exc:    print("=" * 60)    print(f"FAIL - numpy/pandas ABI is broken: {type(exc).__name__}: {exc}")    print("       'numpy.dtype size changed' means pandas' C extension was")    print("       compiled against a different numpy than the one installed.")    raise SystemExit("numpy/pandas ABI broken")# --- The kernel must not be serving a stale transformers ---------------------if "transformers" in sys.modules:    in_memory = getattr(sys.modules["transformers"], "__version__", None)    if str(in_memory) != str(after["transformers"]):        print("=" * 60)        print(f"FAIL - kernel holds transformers {in_memory}, disk has "              f"{after['transformers']}. pip cannot replace an imported module.")        raise SystemExit("stale transformers in kernel")print()print(f"proceeding with {len(installed)} of {len(WANTED)} packages: "      f"{', '.join(installed)}")print()import warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")CONTEXT = 512PRED_LEN = 10# Resolved on the second Kaggle run and recorded in configs/models.yaml and# registry.py. The entry point is TimesFM3Forecaster; timesfm.TimesFm was a# guess and does not exist. The observed namespace was:#   ForecastConfig, TimesFM3Forecaster, TimesFM3Torch, TimesFM_2p5_200M_torch,#   configs, timesfm_2p5, timesfm_2p5_torch, torchTIMESFM_ENTRY_POINT = "TimesFM3Forecaster"def load_series(n):    """One univariate close series; these four take a series, not OHLCV."""    try:        import yfinance as yf        frame = yf.download("AAPL", period="3y", progress=False, auto_adjust=True)        if len(frame):            if isinstance(frame.columns, pd.MultiIndex):                frame.columns = frame.columns.get_level_values(0)            column = "Close" if "Close" in frame else "close"            return frame[column].dropna().tail(n).to_numpy(), "yfinance"    except Exception:        pass    rng = np.random.default_rng(20260902)    return 100 * np.exp(np.cumsum(rng.normal(0, 0.015, n))), "synthetic"values, source = load_series(CONTEXT)print(f"context: {len(values)} sessions from {source}\n")results = {}def attempt(name, checkpoint, fn):    try:        outcome = fn()        results[name] = ("LOADED", str(outcome))        print(f"  {name:<14} LOADED   {checkpoint}  -> {outcome}")    except Exception as exc:        detail = f"{type(exc).__name__}: {exc}"        results[name] = ("FAILED", detail)        print(f"  {name:<14} FAILED   {checkpoint}")        print(f"                 {detail[:200]}")def run_chronos():    import torch    from chronos import BaseChronosPipeline    pipe = BaseChronosPipeline.from_pretrained(        "amazon/chronos-2",        device_map="cuda" if torch.cuda.is_available() else "cpu")    out = pipe.predict(torch.tensor(values, dtype=torch.float32), PRED_LEN)    return tuple(np.asarray(out).shape)def run_timesfm():    """Use the entry point the last run identified; still print the namespace.    If a future release moves it again, the print is what makes that visible    immediately instead of after another wasted session.    """    import timesfm    print(f"    timesfm version  {getattr(timesfm, '__version__', 'unknown')}")    public = sorted(n for n in dir(timesfm) if not n.startswith("_"))    print(f"    dir(timesfm)     {public}")    if not hasattr(timesfm, TIMESFM_ENTRY_POINT):        raise AttributeError(            f"timesfm.{TIMESFM_ENTRY_POINT} is gone; namespace printed above. "            "Update TIMESFM_ENTRY_POINT here and in configs/models.yaml.")    cls = getattr(timesfm, TIMESFM_ENTRY_POINT)    print(f"    entry point      timesfm.{TIMESFM_ENTRY_POINT}")    model = cls()    for loader in ("load_checkpoint", "from_pretrained", "compile"):        if hasattr(model, loader):            try:                getattr(model, loader)()                print(f"    called           .{loader}()")            except Exception as exc:                print(f"    .{loader}() raised {type(exc).__name__}: {str(exc)[:90]}")    return f"constructed via timesfm.{TIMESFM_ENTRY_POINT}"def run_moirai():    from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module    module = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")    forecast = Moirai2Forecast(module=module, prediction_length=PRED_LEN,                               context_length=len(values),                               target_dim=1, feat_dynamic_real_dim=0,                               past_feat_dynamic_real_dim=0)    predictor = forecast.create_predictor(batch_size=1)    return f"constructed {type(predictor).__name__}"def run_ttm():    import torch    from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction    model = TinyTimeMixerForPrediction.from_pretrained(        "ibm-granite/granite-timeseries-ttm-r2")    ctx = model.config.context_length    x = torch.tensor(values[-ctx:], dtype=torch.float32).reshape(1, ctx, 1)    with torch.no_grad():        out = model(past_values=x)    return tuple(out.prediction_outputs.shape)print("=" * 60)CHECKPOINTS = {    "Chronos-2": ("amazon/chronos-2", run_chronos),    "TimesFM-2.5": ("google/timesfm-2.5-200m-pytorch", run_timesfm),    "Moirai-2.0": ("Salesforce/moirai-2.0-R-small", run_moirai),    "TTM": ("ibm-granite/granite-timeseries-ttm-r2", run_ttm),}for model, (checkpoint, runner) in CHECKPOINTS.items():    package = PACKAGE_FOR_MODEL[model]    if package in installed:        attempt(model, checkpoint, runner)    else:        results[model] = ("SKIPPED", f"{package} did not install")        print(f"  {model:<14} SKIPPED  {package} did not install under the "              f"constraint; not a statement about the checkpoint")print("=" * 60)final = versions()drifted = [n for n in PIN if after[n] != final[n]]if drifted:    print(f"WARNING: {', '.join(drifted)} moved during the model loads themselves")loaded = [k for k, (s, _) in results.items() if s == "LOADED"]broke = [k for k, (s, _) in results.items() if s == "FAILED"]skipped = [k for k, (s, _) in results.items() if s == "SKIPPED"]print(f"torch {final['torch']} | numpy {final['numpy']} | "      f"transformers {final['transformers']}")print(f"loaded  ({len(loaded)}): {', '.join(loaded) if loaded else 'none'}")print(f"failed  ({len(broke)}): {', '.join(broke) if broke else 'none'}")print(f"skipped ({len(skipped)}): {', '.join(skipped) if skipped else 'none'}")if broke:    print(f"FAIL - {len(broke)} loaded package(s) could not run: {', '.join(broke)}")elif skipped:    print(f"PARTIAL - {len(loaded)} of {len(CHECKPOINTS)} loaded; "          f"{len(skipped)} skipped for install failure")else:    print("PASS - all four loaded")print()print("Reporting only. Do not debug here - record what broke, then decide")print("whether the integration cost is worth the GPU budget.")

In [ ]:
# ============================================================# CELL 4 - NEURALFORECAST SMOKE TEST (DLinear, MQLoss, 9-point grid)# ============================================================# RUN THE REAL NEURAL TIER IN ITS OWN KAGGLE SESSION.## Lightning forks worker processes, and it cannot fork once CUDA has been# initialised in the interpreter. Cells 2 and 3 both initialise it, so running# this cell after them raised:##     RuntimeError: Lightning can't create new processes if CUDA is already#     initialized## The first attempt at this passed num_workers_loader=0, which failed with##     TypeError: Trainer.__init__() got an unexpected keyword argument#     'num_workers_loader'## because that parameter does not exist in neuralforecast 3.2.1. Anything the# model constructor does not name falls into **trainer_kwargs and is handed# straight to pl.Trainer, which rejects it. Verified against the 3.2.1 source,# not assumed: `grep -rn num_workers_loader neuralforecast/` returns nothing,# and DLinear.__init__ names `dataloader_kwargs=None` instead. The chain is##     DLinear(dataloader_kwargs={"num_workers": 0})#       -> BaseModel.dataloader_kwargs#       -> TimeSeriesDataModule(**dataloader_kwargs)#       -> the DataLoader## while accelerator and devices legitimately travel through **trainer_kwargs to# pl.Trainer. So: dataloader settings in dataloader_kwargs, trainer settings# loose. Zero workers is a workaround, not a fix - it serialises data loading,# which is fine for one epoch on five tickers and wrong for seven models over# 50 tickers and four folds. The foundation tier and the neural tier belong in# separate sessions: a fresh kernel per tier, so the neural run gets real# dataloader workers and an uninitialised CUDA context.# ============================================================import osimport warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "JNJ"]HORIZON = 1LOOKBACK = 60# The grid must match configs/eval.yaml exactly - training.quantiles and the# evaluation grid are checked for equality at load time by# sp500bench.config.load_configs. If the repo is not on this machine the literal# is used, and the source is printed so a silent divergence is visible.GRID = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]grid_source = "literal fallback (repo not found)"for candidate in ("configs/eval.yaml", "../configs/eval.yaml",                  "/kaggle/working/sp500-forecast-benchmark/configs/eval.yaml"):    if os.path.exists(candidate):        import yaml        GRID = yaml.safe_load(open(candidate))["quantiles"]        grid_source = candidate        breakprint(f"quantile grid ({len(GRID)}): {GRID}")print(f"source: {grid_source}\n")def load_panel(tickers):    try:        import yfinance as yf        frame = yf.download(tickers, period="3y", progress=False, auto_adjust=True)        close = frame["Close"] if "Close" in frame else frame["close"]        long = (close.stack().rename("price").reset_index())        long.columns = ["ds", "unique_id", "price"]        long["y"] = long.groupby("unique_id")["price"].transform(            lambda s: np.log(s).diff())        return long.dropna()[["unique_id", "ds", "y"]], "yfinance"    except Exception:        rng = np.random.default_rng(20260902)        dates = pd.bdate_range("2023-01-02", periods=600)        rows = [pd.DataFrame({"unique_id": t, "ds": dates,                              "y": rng.normal(0, 0.018, len(dates))})                for t in tickers]        return pd.concat(rows, ignore_index=True), "synthetic"print("=" * 60)try:    from neuralforecast import NeuralForecast    from neuralforecast.losses.pytorch import MQLoss    from neuralforecast.models import DLinear    panel, source = load_panel(TICKERS)    print(f"panel              {source}, {len(panel):,} rows, "          f"{panel.unique_id.nunique()} tickers")    import torch    on_gpu = torch.cuda.is_available()    model = DLinear(        h=HORIZON, input_size=LOOKBACK, loss=MQLoss(quantiles=GRID),        max_steps=1, val_check_steps=1, enable_progress_bar=False,        logger=False, scaler_type="standard",        # No forked dataloader workers. In 3.2.1 this is dataloader_kwargs,        # NOT num_workers_loader - see the note at the top of the cell.        dataloader_kwargs={"num_workers": 0},        # Pin to one device explicitly rather than letting Lightning choose a        # strategy; "auto" can select a spawning one on a multi-GPU box.        accelerator="gpu" if on_gpu else "cpu",        devices=1,    )    import neuralforecast    print(f"neuralforecast     {neuralforecast.__version__}")    print(f"trainer            accelerator={'gpu' if on_gpu else 'cpu'}, devices=1")    print('dataloader         dataloader_kwargs={"num_workers": 0}')    nf = NeuralForecast(models=[model], freq="B")    nf.fit(df=panel, verbose=False)    forecast = nf.predict()    columns = list(forecast.columns)    print(f"forecast shape     {forecast.shape}")    print(f"forecast columns   {columns}")    # Do not assume the naming scheme - count what actually came back. Different    # neuralforecast versions spell MQLoss outputs differently (-q-1.0, -lo-98,    # -median), and the evaluation layer has to read whatever this is.    output_columns = [c for c in columns if c not in ("unique_id", "ds")]    print(f"\n{len(output_columns)} output column(s) for {len(GRID)} quantiles:")    for c in output_columns:        print(f"  {c}")    emitted = len(output_columns)    print("=" * 60)    if emitted == len(GRID):        print(f"PASS - DLinear emitted all {len(GRID)} quantile columns")        print("       Reminder: run the real neural tier in a fresh session.")    else:        print(f"FAIL - expected {len(GRID)} quantile columns, got {emitted}")        print("       Map these names to the q0.01..q0.99 convention in")        print("       sp500bench.config.quantile_column before any real run.")except Exception as exc:    import traceback    traceback.print_exc()    print("=" * 60)    print(f"FAIL - neuralforecast: {type(exc).__name__}: {exc}")